In [ ]:
# notebooks/03_Test.ipynb

# 1. Environment Initialization
import os
import shutil
import sys
import random
import itertools
import numpy as np
import pandas as pd
import torch
from torch.amp import autocast
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install -q monai
from monai.inferers import sliding_window_inference

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)

import src.config as config
from src.dataset import get_test_dataloader
from src.metrics import SegmentationMetrics
from src.models.mamba_backbone import MambaBackbone, SharedDeepMambaBackbone
from src.models.fusion import PresenceAwareCrossModalFusion
from src.models.decoder import SegmentationDecoder3D, AuxiliaryDecoder3D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_ABC.zip"
LOCAL_EXTRACT_DIR = "/content/MICCAI_BraTS2020_TrainingData_ABC"
LOCAL_DATA_DIR = LOCAL_EXTRACT_DIR

# Verify archive existence immediately before runtime allocation
assert os.path.exists(DATASET_ZIP), f"Dataset archive not found: {DATASET_ZIP}"

# Robust extraction guard: triggers if directory does not exist or is completely empty
if not os.path.exists(LOCAL_DATA_DIR) or len(os.listdir(LOCAL_DATA_DIR)) == 0:
    print(f"Extracting preprocessed dataset to local runtime storage: {LOCAL_EXTRACT_DIR}...")
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    shutil.unpack_archive(DATASET_ZIP, LOCAL_EXTRACT_DIR, "zip")
    print("Extraction complete.")
else:
    print("Existing local preprocessed dataset detected. Skipping extraction.")

In [ ]:
# 2. Pipeline Dataset Loaders Construction
test_loader = get_test_dataloader()

# 3. Structural Module Instantiations & Checkpoint Loading
backbone = MambaBackbone(embed_dim=config.EMBED_DIM).to(device)
fusion = PresenceAwareCrossModalFusion(embed_dim=config.EMBED_DIM).to(device)
shared_backbone = SharedDeepMambaBackbone(embed_dim=config.EMBED_DIM).to(device)
decoder = SegmentationDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)
aux_decoder = AuxiliaryDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)

best_seg_path = os.path.join(config.CHECKPOINT_DIR, "best_seg.pth")
assert os.path.exists(best_seg_path), f"Checkpoint not found: {best_seg_path}"

checkpoint = torch.load(best_seg_path, map_location=device)
backbone.load_state_dict(checkpoint["backbone_state"])
fusion.load_state_dict(checkpoint["fusion_state"])
shared_backbone.load_state_dict(checkpoint["shared_backbone_state"])
decoder.load_state_dict(checkpoint["decoder_state"])
aux_decoder.load_state_dict(checkpoint["aux_decoder_state"])

backbone.eval()
fusion.eval()
shared_backbone.eval()
decoder.eval()

In [ ]:
# 4. Define 15 Modality Combinations
# 0: T1, 1: T1ce, 2: T2, 3: FLAIR
mod_idx = [0, 1, 2, 3]
mod_names = ["T1", "T1ce", "T2", "FLAIR"]

combinations = []
for r in range(1, 5):
    combinations.extend(list(itertools.combinations(mod_idx, r)))

# --- Resumption and Directory Setup ---
csv_out_path = os.path.join(config.RESULT_DIR, "modality_dropout_results.csv")

completed_combinations = []
if os.path.exists(csv_out_path):
    existing_df = pd.read_csv(csv_out_path)
    if "Present Modalities" in existing_df.columns:
        completed_combinations = existing_df["Present Modalities"].tolist()
        print(f"[*] Found existing results. Resuming... ({len(completed_combinations)}/15 completed)")
# --------------------------------------

results = []

In [ ]:
# 5. Evaluation Loop across 15 settings
print(f"Starting multi-modality evaluation across {len(combinations)} settings...")

with torch.no_grad():
    for comb in combinations:
        comb_names = "+".join([mod_names[i] for i in comb])

        # --- Skip if already processed ---
        if comb_names in completed_combinations:
            print(f"Skipping {comb_names} (already completed)...")
            continue
        # ---------------------------------

        print(f"\n--- Testing Modalities: {comb_names} ---")

        keep_mask = torch.zeros(4, device=device)
        keep_mask[list(comb)] = 1.0

        seg_tracker = SegmentationMetrics()

        for batch in tqdm(test_loader, desc=f"Eval {comb_names}", leave=False):
            images = batch["image"].to(device)
            seg_targets = batch["label"].to(device)
            B_current = images.size(0)
            batch_seg_logits = []

            for b in range(B_current):
                single_img = images[b:b+1]

                def evaluation_predictor(patch_images):
                    # 1. Apply dropout to the raw input images FIRST
                    processed_images = patch_images.clone()
                    for i in range(4):
                        if keep_mask[i] == 0.0:
                            processed_images[:, i, :, :, :] = 0.0

                    # 2. Backbone now processes genuinely incomplete inputs
                    modality_tokens, spatial_shape, skip_features, _ = backbone(processed_images)

                    # 3. Format tokens for the fusion module as required
                    processed_modality_tokens = []
                    for i in range(4):
                        if keep_mask[i] == 1.0:
                            processed_modality_tokens.append(modality_tokens[i])
                        else:
                            processed_modality_tokens.append(torch.zeros_like(modality_tokens[i]))

                    fused_tokens = fusion(processed_modality_tokens, processed_images)
                    latent_tokens = shared_backbone(fused_tokens)
                    seg_logits = decoder(latent_tokens, spatial_shape, skip_features)

                    return seg_logits

                with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    seg_logits = sliding_window_inference(
                        inputs=single_img,
                        roi_size=config.PATCH_SIZE,
                        sw_batch_size=1,
                        predictor=evaluation_predictor,
                        overlap=0.5,
                        mode="gaussian"
                    )
                batch_seg_logits.append(seg_logits)

            seg_logits = torch.cat(batch_seg_logits, dim=0)
            seg_preds = torch.argmax(seg_logits, dim=1, keepdim=True)
            seg_tracker.update(seg_preds, seg_targets, run_hd=False)

        metrics = seg_tracker.compute(run_hd=False)
        mean_dice = (metrics["dice_WT"] + metrics["dice_TC"] + metrics["dice_ET"]) / 3.0

        current_result = {
            "Missing Modalities": "+".join([mod_names[i] for i in mod_idx if i not in comb]) or "None",
            "Present Modalities": comb_names,
            "Dice WT": round(metrics["dice_WT"], 4),
            "Dice TC": round(metrics["dice_TC"], 4),
            "Dice ET": round(metrics["dice_ET"], 4),
            "Mean Dice": round(mean_dice, 4),
            "HD95 WT": round(metrics["hd95_WT"], 4) if "hd95_WT" in metrics else "N/A",
            "HD95 TC": round(metrics["hd95_TC"], 4) if "hd95_TC" in metrics else "N/A",
            "HD95 ET": round(metrics["hd95_ET"], 4) if "hd95_ET" in metrics else "N/A"
        }

        results.append(current_result)
        print(f"Mean Dice: {mean_dice:.4f} | WT: {metrics['dice_WT']:.4f}, TC: {metrics['dice_TC']:.4f}, ET: {metrics['dice_ET']:.4f}")

        # --- Incremental Save to CSV ---
        current_df = pd.DataFrame([current_result])
        if not os.path.exists(csv_out_path):
            current_df.to_csv(csv_out_path, index=False)
        else:
            current_df.to_csv(csv_out_path, mode='a', header=False, index=False)
        # -------------------------------

In [ ]:
# 6. Display Results
print("\nTesting finished. Loading final combined results:")
if os.path.exists(csv_out_path):
    df_results = pd.read_csv(csv_out_path)
    display(df_results)
else:
    print("No results found.")